In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from fuzzywuzzy import process
import warnings 

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
warnings.filterwarnings ('ignore')
pd.set_option ('display.width', None)
pd.set_option ('display.max_rows', 100)
pd.set_option ('display.max_columns', 50)

# 1. Create `Football_Player_Profile` Dataset

## a. Read `Transfermarkt.csv` files

In [28]:
def Read_Transfermarkt_Data (league):
    folder_path = f'Web_Scraping_Files/Transfermarkt/{league}'
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    # Đọc từng file vào DataFrame
    list_dfs = []
    for file in csv_files:
        file_path = os.path.join(folder_path, file)  # nối đường dẫn đầy đủ
        df = pd.read_csv(file_path)
        list_dfs.append(df)

    # Gộp lại
    df_all = pd.concat(list_dfs, ignore_index=True)
    df_all = df_all.rename (columns={'Name':'Player'})
    
    df_all.drop (['Unnamed: 0'], inplace=True, axis=1)
    df_all.sort_values (by='CLB', inplace=True)
    
    CLB = df_all ['CLB'].unique ()
    print (CLB)
    return df_all

EPL_Transfermarkt = Read_Transfermarkt_Data  ('EPL')
Bundesliga_Transfermarkt = Read_Transfermarkt_Data  ('Bundesliga')
Laliga_Transfermarkt = Read_Transfermarkt_Data ('Laliga')
SerieA_Transfermarkt = Read_Transfermarkt_Data ('SerieA')
Ligue1_Transfermarkt = Read_Transfermarkt_Data  ('Ligue1')
EPL_Transfermarkt


['AFC Bournemouth' 'Arsenal FC' 'Aston Villa' 'Brentford FC'
 'Brighton & Hove Albion' 'Chelsea FC' 'Crystal Palace' 'Everton FC'
 'Fulham FC' 'Ipswich Town' 'Leicester City' 'Liverpool FC'
 'Manchester City' 'Manchester United' 'Newcastle United'
 'Nottingham Forest' 'Southampton FC' 'Tottenham Hotspur'
 'West Ham United' 'Wolverhampton Wanderers']
['1.FC Heidenheim 1846' '1.FC Union Berlin' '1.FSV Mainz 05'
 'Bayer 04 Leverkusen' 'Bayern Munich' 'Borussia Dortmund'
 'Borussia Mönchengladbach' 'Eintracht Frankfurt' 'FC Augsburg'
 'FC St. Pauli' 'Holstein Kiel' 'RB Leipzig' 'SC Freiburg'
 'SV Werder Bremen' 'TSG 1899 Hoffenheim' 'VfB Stuttgart' 'VfL Bochum'
 'VfL Wolfsburg']
['Athletic Bilbao' 'Atlético de Madrid' 'CA Osasuna' 'CD Leganés'
 'Celta de Vigo' 'Deportivo Alavés' 'FC Barcelona' 'Getafe CF' 'Girona FC'
 'RCD Espanyol Barcelona' 'RCD Mallorca' 'Rayo Vallecano'
 'Real Betis Balompié' 'Real Madrid' 'Real Sociedad' 'Real Valladolid CF'
 'Sevilla FC' 'UD Las Palmas' 'Valencia CF'

,CLB,Player,Main_position,Height,Foot,Contract,Market_Value
149,AFC Bournemouth,Ryan Christie,Defensive Midfield,"1,78m",left,"Jun 30, 2027",€12.00m
160,AFC Bournemouth,Daniel Jebbison,Centre-Forward,"1,90m",right,"Jun 30, 2028",€4.00m
139,AFC Bournemouth,Dean Huijsen,Centre-Back,"1,97m",both,"Jun 30, 2030",€42.00m
159,AFC Bournemouth,Enes Ünal,Centre-Forward,"1,87m",right,"Jun 30, 2028",€13.00m
158,AFC Bournemouth,Evanilson,Centre-Forward,"1,83m",both,-,€35.00m
...,...,...,...,...,...,...,...
260,Wolverhampton Wanderers,Pedro Lima,Right-Back,"1,74m",right,"Jun 30, 2029",€5.00m
259,Wolverhampton Wanderers,Nélson Semedo,Right-Back,"1,77m",right,"Jun 30, 2025",€10.00m
258,Wolverhampton Wanderers,Rayan Aït-Nouri,Left-Back,"1,80m",left,"Jun 30, 2026",€35.00m
267,Wolverhampton Wanderers,Tommy Doyle,Central Midfield,"1,72m",right,"Jun 30, 2028",€10.00m


## b. Read `base_info.csv` files

In [29]:
def basic_info_function (league):
    path = f"Web_Scraping_Files/{league}/base_info_{league}.csv"
    base_info_ = pd.read_csv (path)
    
    # Xoá cột Unnamed: 0 nếu tồn tại
    if 'Unnamed: 0' in base_info_.columns:
        base_info_.drop(columns=['Unnamed: 0'], inplace=True)

    # Reset index lại cho sạch
    base_info_.reset_index(drop=True, inplace=True)
    
    # Đổi tên cột Name -> Player
    base_info_.rename (columns={'Name':'Player'}, inplace=True)
    base_info_.insert (2, 'League',f'{league}')
    
    print (f"{league}: {base_info_.shape}")
    # print (base_info_['Club'].unique ())
    return base_info_

    # info.append (base_info_)
EPL_Basic_info = basic_info_function ('EPL')

Laliga_Basic_info = basic_info_function ('Laliga')

Bundesliga_Basic_info = basic_info_function ('Bundesliga')

SerieA_Basic_info = basic_info_function ('SerieA')

Ligue1_Basic_info = basic_info_function ('Ligue1')

EPL_Basic_info
# base_info = pd.concat (info, ignore_index=True)
# df = df.rename (columns={'ten_cu':'ten_moi'})
# Đổi cột Name --> Player để đồng nhất
# base_info = base_info.rename (columns={'Name':'Player'})
# base_info.drop (columns=['Unnamed: 0'], inplace=True)
# base_info

EPL: (564, 7)
Laliga: (587, 7)
Bundesliga: (487, 7)
SerieA: (625, 7)
Ligue1: (543, 7)


,Player,Club,League,Age,Height,National,Positions
0,Mohamed Salah,Liverpool,EPL,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo..."
1,Bukayo Saka,Arsenal,EPL,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)"
2,Alexander Isak,Newcastle,EPL,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward"
3,Matheus Cunha,Wolves,EPL,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward"
4,Erling Haaland,Manchester City,EPL,24 years old (21-07-2000),194cm,Norway,Forward
...,...,...,...,...,...,...,...
559,Mason Holgate,West Bromwich Albion,EPL,28 years old (22-10-1996),184cm,England,"Defender (Centre, Right), Defensive Midfielder..."
560,Harry Clarke,Sheffield United,EPL,24 years old (02-03-2001),180cm,England,"Defender (Centre, Right), Midfielder (Right)"
561,Jayden Danns,Liverpool,EPL,19 years old (16-01-2006),183cm,England,Forward
562,Ben Godfrey,Ipswich,EPL,27 years old (15-01-1998),184cm,England,"Defender (Centre, Left, Right)"


## c. Standardizing Clubs' names: whoscored -> Transfermarkt

In [30]:
club_name_mapping = {
    # EPL
    'Arsenal': 'Arsenal FC',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'AFC Bournemouth',
    'Brentford': 'Brentford FC',
    'Brighton': 'Brighton & Hove Albion',
    'Chelsea': 'Chelsea FC',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton FC',
    'Fulham': 'Fulham FC',
    'Liverpool': 'Liverpool FC',
    'Manchester City': 'Manchester City',
    'Manchester United': 'Manchester United',
    'Newcastle': 'Newcastle United',
    'Nottingham Forest': 'Nottingham Forest',
    'Tottenham': 'Tottenham Hotspur',
    'West Ham': 'West Ham United',
    'Wolves': 'Wolverhampton Wanderers',
    'Leicester': 'Leicester City',
    'Ipswich': 'Ipswich Town',
    'Southampton': 'Southampton FC',
    # Bundesliga
    'FC Heidenheim': '1.FC Heidenheim 1846',
    'Union Berlin': '1.FC Union Berlin',
    'Mainz 05': '1.FSV Mainz 05',
    'Bayer Leverkusen': 'Bayer 04 Leverkusen',
    'Bayern Munich': 'Bayern Munich',
    'Borussia Dortmund': 'Borussia Dortmund',
    'Borussia M.Gladbach': 'Borussia Mönchengladbach',
    'Eintracht Frankfurt': 'Eintracht Frankfurt',
    'Augsburg': 'FC Augsburg',
    'St. Pauli': 'FC St. Pauli',
    'Holstein Kiel': 'Holstein Kiel',
    'RB Leipzig': 'RB Leipzig',
    'Freiburg': 'SC Freiburg',
    'Werder Bremen': 'SV Werder Bremen',
    'Hoffenheim': 'TSG 1899 Hoffenheim',
    'VfB Stuttgart': 'VfB Stuttgart',
    'Bochum': 'VfL Bochum',
    'Wolfsburg': 'VfL Wolfsburg',
    # Laliga
    'Barcelona': 'FC Barcelona',
    'Real Betis': 'Real Betis Balompié',
    'Osasuna': 'CA Osasuna',
    'Real Madrid': 'Real Madrid',
    'Villarreal': 'Villarreal CF',
    'Mallorca': 'RCD Mallorca',
    'Sevilla': 'Sevilla FC',
    'Rayo Vallecano': 'Rayo Vallecano',
    'Athletic Club': 'Athletic Bilbao',
    'Getafe': 'Getafe CF',
    'Espanyol': 'RCD Espanyol Barcelona',
    'Real Sociedad': 'Real Sociedad',
    'Atletico Madrid': 'Atlético de Madrid',
    'Leganes': 'CD Leganés',
    'Valencia': 'Valencia CF',
    'Las Palmas': 'UD Las Palmas',
    'Girona': 'Girona FC',
    'Deportivo Alaves': 'Deportivo Alavés',
    'Celta Vigo': 'Celta de Vigo',
    'Real Valladolid': 'Real Valladolid CF',
    # Ligue1
    'Auxerre': 'AJ Auxerre',
    'Monaco': 'AS Monaco',
    'Saint-Etienne': 'AS Saint-Étienne',
    'Angers': 'Angers SCO',
    'Nantes': 'FC Nantes',
    'Toulouse': 'FC Toulouse',
    'Lille': 'LOSC Lille',
    'Le Havre': 'Le Havre AC',
    'Montpellier': 'Montpellier HSC',
    'Nice': 'OGC Nice',
    'Lyon': 'Olympique Lyon',
    'Marseille': 'Olympique Marseille',
    'Paris Saint-Germain': 'Paris Saint-Germain',
    'Lens': 'RC Lens',
    'Strasbourg': 'RC Strasbourg Alsace',
    'Brest': 'Stade Brestois 29',
    'Reims': 'Stade Reims',
    'Rennes': 'Stade Rennais FC',
    # SerieA
    'AC Milan': 'AC Milan',
    'Monza': 'AC Monza',
    'Fiorentina': 'ACF Fiorentina',
    'Roma': 'AS Roma',
    'Atalanta': 'Atalanta BC',
    'Bologna': 'Bologna FC 1909',
    'Cagliari': 'Cagliari Calcio',
    'Como': 'Como 1907',
    'Empoli': 'FC Empoli',
    'Genoa': 'Genoa CFC',
    'Verona': 'Hellas Verona',
    'Inter': 'Inter Milan',
    'Juventus': 'Juventus FC',
    'Lazio': 'SS Lazio',
    'Napoli': 'SSC Napoli',
    'Torino': 'Torino FC',
    'Lecce': 'US Lecce',
    'Udinese': 'Udinese Calcio',
    'Venezia': 'Venezia FC',
    'Parma Calcio 1913': 'Parma Calcio 1913'
}
def standardize_club_names(df, column='Club', mapping=club_name_mapping):
    df[column] = df[column].replace(mapping)
    return df

EPL_Basic_info = standardize_club_names (EPL_Basic_info)

Laliga_Basic_info = standardize_club_names (Laliga_Basic_info)

Bundesliga_Basic_info = standardize_club_names (Bundesliga_Basic_info)

SerieA_Basic_info = standardize_club_names (SerieA_Basic_info)

Ligue1_Basic_info = standardize_club_names (Ligue1_Basic_info)
EPL_Basic_info


,Player,Club,League,Age,Height,National,Positions
0,Mohamed Salah,Liverpool FC,EPL,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo..."
1,Bukayo Saka,Arsenal FC,EPL,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)"
2,Alexander Isak,Newcastle United,EPL,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward"
3,Matheus Cunha,Wolverhampton Wanderers,EPL,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward"
4,Erling Haaland,Manchester City,EPL,24 years old (21-07-2000),194cm,Norway,Forward
...,...,...,...,...,...,...,...
559,Mason Holgate,West Bromwich Albion,EPL,28 years old (22-10-1996),184cm,England,"Defender (Centre, Right), Defensive Midfielder..."
560,Harry Clarke,Sheffield United,EPL,24 years old (02-03-2001),180cm,England,"Defender (Centre, Right), Midfielder (Right)"
561,Jayden Danns,Liverpool FC,EPL,19 years old (16-01-2006),183cm,England,Forward
562,Ben Godfrey,Ipswich Town,EPL,27 years old (15-01-1998),184cm,England,"Defender (Centre, Left, Right)"


## d. Create match_name columns as a linking key between 2 datasets

In [31]:
from rapidfuzz import process

# Tạo các dictionary tên từ mỗi giải đấu
EPL_choices = EPL_Basic_info['Player'].tolist()
Laliga_choices = Laliga_Basic_info['Player'].tolist()
Bundesliga_choices = Bundesliga_Basic_info['Player'].tolist()
SerieA_choices = SerieA_Basic_info['Player'].tolist()
Ligue1_choices = Ligue1_Basic_info['Player'].tolist()

# Danh sách các DataFrame và choices tương ứng
league_info = {
    'EPL': {'df': EPL_Basic_info, 'choices': EPL_choices},
    'La Liga': {'df': Laliga_Basic_info, 'choices': Laliga_choices},
    'Bundesliga': {'df': Bundesliga_Basic_info, 'choices': Bundesliga_choices},
    'Serie A': {'df': SerieA_Basic_info, 'choices': SerieA_choices},
    'Ligue 1': {'df': Ligue1_Basic_info, 'choices': Ligue1_choices}
}
# Ánh xạ tên gần nhất từ df2
"""
def fuzzy_match(name):
    result = process.extractOne(name, choices, score_cutoff=85)
    if result is not None:
        match, score, _ = result
        return match
    return None
=> Không tối ưu sàng lọc CLB
"""

def fuzzy_match_with_club_check(name, club):
    best_match = None
    
    for league, data in league_info.items():
        choices = data['choices']
        df = data['df']
        
        # Tìm tên cầu thủ gần nhất trong từng giải đấu
        result = process.extractOne(name, choices, score_cutoff=85)
        if result is not None:
            match_name, score, _ = result
            
            # Tìm câu lạc bộ của cầu thủ matched từ df
            matched_row = df[df['Player'] == match_name]
            if not matched_row.empty:
                club_df1 = matched_row.iloc[0]['Club']
                if club_df1 == club:
                    best_match = match_name
                    break  # Chỉ cần tìm thấy 1 sự match tốt nhất, thoát vòng lặp
    return best_match
'''
def fuzzy_match_with_club_check(name, club):
    result = process.extractOne(name, EPL_choices, score_cutoff=85)
    if result is not None:
        match_name, score, _ = result

        # Tìm lại club của cầu thủ matched từ df1
        matched_row = EPL_Basic_info[EPL_Basic_info['Player'] == match_name]
        if not matched_row.empty:
            club_df1 = matched_row.iloc[0]['Club']  # không strip/lower
            if club_df1 == club:
                return match_name
    return None
'''
# Áp dụng hàm cho tất cả các cầu thủ trong EPL_Transfermarkt
EPL_Transfermarkt['matched_name'] = EPL_Transfermarkt.apply(
    lambda row: fuzzy_match_with_club_check(row['Player'], row['CLB']),
    axis=1
)
Laliga_Transfermarkt['matched_name'] = Laliga_Transfermarkt.apply(
    lambda row: fuzzy_match_with_club_check(row['Player'], row['CLB']),
    axis=1
)
Bundesliga_Transfermarkt['matched_name'] = Bundesliga_Transfermarkt.apply(
    lambda row: fuzzy_match_with_club_check(row['Player'], row['CLB']),
    axis=1
)
SerieA_Transfermarkt['matched_name'] = SerieA_Transfermarkt.apply(
    lambda row: fuzzy_match_with_club_check(row['Player'], row['CLB']),
    axis=1
)
Ligue1_Transfermarkt['matched_name'] = Ligue1_Transfermarkt.apply(
    lambda row: fuzzy_match_with_club_check(row['Player'], row['CLB']),
    axis=1
)
EPL_Transfermarkt
# df_all['matched_name'] = df_all['Player'].apply(fuzzy_match)
# df_all


,CLB,Player,Main_position,Height,Foot,Contract,Market_Value,matched_name
149,AFC Bournemouth,Ryan Christie,Defensive Midfield,"1,78m",left,"Jun 30, 2027",€12.00m,Ryan Christie
160,AFC Bournemouth,Daniel Jebbison,Centre-Forward,"1,90m",right,"Jun 30, 2028",€4.00m,Daniel Jebbison
139,AFC Bournemouth,Dean Huijsen,Centre-Back,"1,97m",both,"Jun 30, 2030",€42.00m,Dean Huijsen
159,AFC Bournemouth,Enes Ünal,Centre-Forward,"1,87m",right,"Jun 30, 2028",€13.00m,Enes Ünal
158,AFC Bournemouth,Evanilson,Centre-Forward,"1,83m",both,-,€35.00m,Evanilson
...,...,...,...,...,...,...,...,...
260,Wolverhampton Wanderers,Pedro Lima,Right-Back,"1,74m",right,"Jun 30, 2029",€5.00m,Pedro Lima
259,Wolverhampton Wanderers,Nélson Semedo,Right-Back,"1,77m",right,"Jun 30, 2025",€10.00m,Nélson Semedo
258,Wolverhampton Wanderers,Rayan Aït-Nouri,Left-Back,"1,80m",left,"Jun 30, 2026",€35.00m,Rayan Aït-Nouri
267,Wolverhampton Wanderers,Tommy Doyle,Central Midfield,"1,72m",right,"Jun 30, 2028",€10.00m,Tommy Doyle


## e. Create merged_df per league

In [32]:
# Bước 1: Tạo cột chỉ số gốc để lưu thứ tự gốc của EPL
EPL_Basic_info['orig_index'] = EPL_Basic_info.index
Laliga_Basic_info['orig_index'] = Laliga_Basic_info.index
Bundesliga_Basic_info['orig_index'] = Bundesliga_Basic_info.index
SerieA_Basic_info['orig_index'] = SerieA_Basic_info.index
Ligue1_Basic_info['orig_index'] = Ligue1_Basic_info.index

# Bước 2: Merge như trước
def merged_dataframe (Basic_info, Transfermarkt):
    merged_df = pd.merge(
        Basic_info, Transfermarkt,
        left_on='Player',
        right_on='matched_name',
        how='outer',
        suffixes=('_EPL', '_all')
    )
    # Bước 3: Dựa vào cột 'orig_index', sắp xếp lại theo thứ tự EPL (những cầu thủ không match thì orig_index là NaN)
    merged_df.sort_values(by='orig_index', na_position='last', inplace=True)
    # Bước 4: Xoá cột chỉ số nếu không cần giữ lại
    merged_df.drop(columns='orig_index', inplace=True)
    merged_df.reset_index(drop=True, inplace=True)
    return merged_df

# Bước 3: Dựa vào cột 'orig_index', sắp xếp lại theo thứ tự EPL (những cầu thủ không match thì orig_index là NaN)
EPL_merged_df = merged_dataframe (EPL_Basic_info, EPL_Transfermarkt)
Laliga_merged_df = merged_dataframe (Laliga_Basic_info, Laliga_Transfermarkt)
Bundesliga_merged_df = merged_dataframe (Bundesliga_Basic_info, Bundesliga_Transfermarkt)
SerieA_merged_df = merged_dataframe (SerieA_Basic_info, SerieA_Transfermarkt)
Ligue1_merged_df = merged_dataframe (Ligue1_Basic_info, Ligue1_Transfermarkt)

Ligue1_merged_df

,Player_EPL,Club,League,Age,Height_EPL,National,Positions,CLB,Player_all,Main_position,Height_all,Foot,Contract,Market_Value,matched_name
0,Benjamin Bourigeaud,Stade Rennais FC,Ligue1,31 years old (14-01-1994),178cm,France,"Midfielder (Centre, Left, Right)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Logan Costa,Villarreal CF,Ligue1,24 years old (01-04-2001),190cm,France,Defender (Centre),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ousmane Dembélé,Paris Saint-Germain,Ligue1,27 years old (15-05-1997),178cm,France,"Attacking Midfielder (Centre, Left, Right), Fo...",Paris Saint-Germain,Ousmane Dembélé,Right Winger,"1,78m",both,"Jun 30, 2028",€75.00m,Ousmane Dembélé
3,Ismail Jakobs,Galatasaray,Ligue1,25 years old (17-08-1999),184cm,Senegal,"Defender (Left), Midfielder (Left)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Amine Gouiri,Olympique Marseille,Ligue1,25 years old (16-02-2000),180cm,France,"Attacking Midfielder (Centre, Left, Right), Fo...",Olympique Marseille,Amine Gouiri,Centre-Forward,"1,81m",right,"Jun 30, 2029",€20.00m,Amine Gouiri
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stade Reims,Alexandre Olliero,Goalkeeper,"1,93m",right,"Jun 30, 2026",€300k,None
582,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stade Reims,Ludovic Butelle,Goalkeeper,"1,88m",left,"Jun 30, 2025",€100k,None
583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stade Rennais FC,Geoffrey Lembet,Goalkeeper,"1,86m",left,"Jun 30, 2025",€150k,None
584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stade Rennais FC,Doğan Alemdar,Goalkeeper,"1,92m",right,"Jun 30, 2027",€2.00m,None


> Combine 2 columns Player_EPL and Player_all

In [33]:
EPL_merged_df['Player'] = EPL_merged_df['Player_EPL'].combine_first(EPL_merged_df['Player_all'])
Laliga_merged_df['Player'] = Laliga_merged_df['Player_EPL'].combine_first(Laliga_merged_df['Player_all'])
Bundesliga_merged_df['Player'] = Bundesliga_merged_df['Player_EPL'].combine_first(Bundesliga_merged_df['Player_all'])
SerieA_merged_df['Player'] = SerieA_merged_df['Player_EPL'].combine_first(SerieA_merged_df['Player_all'])
Ligue1_merged_df['Player'] = Ligue1_merged_df['Player_EPL'].combine_first(Ligue1_merged_df['Player_all'])

EPL_merged_df['Club'] = EPL_merged_df['Club'].combine_first(EPL_merged_df['CLB'])
Laliga_merged_df['Club'] = Laliga_merged_df['Club'].combine_first(Laliga_merged_df['CLB'])
Bundesliga_merged_df['Club'] = Bundesliga_merged_df['Club'].combine_first(Bundesliga_merged_df['CLB'])
SerieA_merged_df['Club'] = SerieA_merged_df['Club'].combine_first(SerieA_merged_df['CLB'])
Ligue1_merged_df['Club'] = Ligue1_merged_df['Club'].combine_first(Ligue1_merged_df['CLB'])

## f. Read `data_summary_EPL.csv` files

In [34]:
def Data_Summary (league):
    path = f"Web_Scraping_Files/{league}/data_summary_{league}.csv"
    data_sum = pd.read_csv (path)
    data_sum.drop (columns=['Player'], inplace=True)
    if 'Unnamed: 0' in data_sum.columns:
        data_sum.drop(columns=['Unnamed: 0'], inplace=True)
    
    print (f"{league} data summary shape: {data_sum.shape}")
    return data_sum
        
EPL_data_summary = Data_Summary ('EPL')
EPL_SpG = EPL_data_summary ["SpG"]
EPL_data_summary.drop ('SpG', axis=1, inplace=True)
EPL_SpG = pd.DataFrame (EPL_SpG)

Laliga_data_summary = Data_Summary ('Laliga')
Laliga_SpG = Laliga_data_summary ["SpG"]
Laliga_data_summary.drop ('SpG', axis=1, inplace=True)
Laliga_SpG = pd.DataFrame (Laliga_SpG)

Bundesliga_data_summary = Data_Summary ('Bundesliga')
Bundesliga_SpG = Bundesliga_data_summary ["SpG"]
Bundesliga_data_summary.drop ('SpG', axis=1, inplace=True)
Bundesliga_SpG = pd.DataFrame (Bundesliga_SpG)

SerieA_data_summary = Data_Summary ('SerieA')
SerieA_SpG = SerieA_data_summary ["SpG"]
SerieA_data_summary.drop ('SpG', axis=1, inplace=True)
SerieA_SpG = pd.DataFrame (SerieA_SpG)

Ligue1_data_summary = Data_Summary ('Ligue1')
Ligue1_SpG = Ligue1_data_summary ["SpG"]
Ligue1_data_summary.drop ('SpG', axis=1, inplace=True)
Ligue1_SpG = pd.DataFrame (Ligue1_SpG)

Ligue1_data_summary

EPL data summary shape: (564, 11)
Laliga data summary shape: (587, 11)
Bundesliga data summary shape: (487, 11)
SerieA data summary shape: (625, 11)
Ligue1 data summary shape: (543, 11)


,Apps,Mins,Goals,Assists,Yel,Red,PS%,AerialsWon,MotM,Rating
0,1,81,1,-,-,-,85.7,2,-,7.99
1,1,90,-,-,-,-,83.3,4,-,7.73
2,19(9),1663,21,6,2,-,83.3,0.1,9,7.70
3,2,180,-,-,-,-,88.5,0.5,-,7.55
4,9(2),789,7,3,-,-,89.7,0.2,2,7.44
...,...,...,...,...,...,...,...,...,...,...
538,0(1),13,-,-,-,-,100,-,-,5.87
539,0(2),19,-,-,1,-,66.7,-,-,5.79
540,6,413,-,-,2,1,88.6,1,-,5.76
541,0(1),26,-,-,-,-,75,-,-,5.67


## g. Create `Football_Player_Profile.csv`

In [35]:
# Football_Player_Profile = pd.merge (base_info, data_summary, on='Player')
EPL_Football_Player_Profile = pd.concat([EPL_merged_df, EPL_data_summary.reindex(EPL_merged_df.index)], axis=1)
Laliga_Football_Player_Profile = pd.concat([Laliga_merged_df, Laliga_data_summary.reindex(Laliga_merged_df.index)], axis=1)
Bundesliga_Football_Player_Profile = pd.concat([Bundesliga_merged_df, Bundesliga_data_summary.reindex(Bundesliga_merged_df.index)], axis=1)
SerieA_Football_Player_Profile = pd.concat([SerieA_merged_df, SerieA_data_summary.reindex(SerieA_merged_df.index)], axis=1)
Ligue1_Football_Player_Profile = pd.concat([Ligue1_merged_df, Ligue1_data_summary.reindex(Ligue1_merged_df.index)], axis=1)

# XOÁ CÁC THUỘC TÍNH KHÔNG CẦN THIẾT
def Design_Football_Player_Profile (df, league):
    df['Height_all'].fillna(df['Height_EPL'], inplace=True)
    df.drop (columns = ['Player_EPL','Height_EPL','CLB','Player_all','matched_name'], inplace=True)
    df = df.reindex (columns=[
        'Player','Club','League','Age','Height_all','National',
        'Positions','Main_position','Foot','Apps','Mins',
        'Goals', 'Assists', 'Yel','Red','PS%','AerialsWon',
        'MotM','Rating','Contract','Market_Value'
    ])
    df['League'].fillna (league, inplace=True)
    return df

EPL_Football_Player_Profile = Design_Football_Player_Profile (EPL_Football_Player_Profile, 'EPL')
Laliga_Football_Player_Profile = Design_Football_Player_Profile (Laliga_Football_Player_Profile, 'Laliga')
Bundesliga_Football_Player_Profile = Design_Football_Player_Profile (Bundesliga_Football_Player_Profile, 'Bundesliga')
SerieA_Football_Player_Profile = Design_Football_Player_Profile (SerieA_Football_Player_Profile, 'SerieA')
Ligue1_Football_Player_Profile = Design_Football_Player_Profile (Ligue1_Football_Player_Profile, 'Ligue1')

# Tạo file ..._Football_Player_Profile.csv
EPL_Football_Player_Profile.to_csv ('EPL_Football_Player_Profile.csv')
Laliga_Football_Player_Profile.to_csv ('Laliga_Football_Player_Profile.csv')
Bundesliga_Football_Player_Profile.to_csv ('Bundesliga_Football_Player_Profile.csv')
SerieA_Football_Player_Profile.to_csv ('SerieA_Football_Player_Profile.csv')
Ligue1_Football_Player_Profile.to_csv ('Ligue1_Football_Player_Profile.csv')
Football_Player_Profile_Full = pd.concat ([EPL_Football_Player_Profile, Laliga_Football_Player_Profile,
                                           Bundesliga_Football_Player_Profile, SerieA_Football_Player_Profile,
                                           Ligue1_Football_Player_Profile
                                        ], axis=0, ignore_index=True)
Football_Player_Profile_Full.to_csv ('Football_Player_Profile_Full.csv')

# 2. Create `Offensive_Profile` Dataset

__Lưu ý:__ Đây là dữ liệu thu thập từ WhoScored

In [36]:
def Read_Data_Details_Offensive (league, SpG):
    # Read *data_details_Goals_EPL.csv
    Goals = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Goals_{league}.csv')
    Goals = Goals.reindex (columns =[
                                'Total','OutOfBox',
                                'SixYardBox','PenaltyArea'
                            ])
    Goals = Goals.rename (columns = {
                        'Total':'TotGs',
                        'OutOfBox':'OOGBs',
                        'SixYardBox':'SYBGs',
                        'PenaltyArea':'PAGs'
                    })
    if 'Unnamed: 0' in Goals.columns:
        Goals.drop(columns=['Unnamed: 0'], inplace=True)
        
    SpG = SpG.reset_index (drop=True)
    # Gộp thuộc tính SpG 
    Goals = pd.concat ([SpG,Goals], axis=1)
        
    # Read *data_details_Shots_EPL.csv
    Shots = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Shots_{league}.csv') 
    Shots = Shots.rename (columns = {
        'Total' : 'TotSh',
        'OutOfBox' : 'OOBSh',
        'SixYardBox' : 'SYBSh',
        'PenaltyArea' : 'PASh'
    })
    if 'Unnamed: 0' in Shots.columns:
        Shots.drop(columns=['Unnamed: 0'], inplace=True)
    
    # Read *data_details_Dribbles_EPL.csv
    Dribbles = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Dribbles_{league}.csv') 
    Dribbles = Dribbles [['Total Dribbles','Unsuccessful','Successful']]
    Dribbles.rename (columns={
        'Total Dribbles' : 'TotDrib',
        'Unsuccessful' : 'UnDrib',
        'Successful' : 'SucDrib'
    }, inplace=True)
    if 'Unnamed: 0' in Dribbles.columns:
        Dribbles.drop(columns=['Unnamed: 0'], inplace=True)
    
    # Read *data_details_Possession loss_EPL.csv
    Possession_Loss = pd.read_csv (f"Web_Scraping_Files/{league}/data_details_Possession loss_{league}.csv")
    Possession_Loss.rename (columns = {
            'UnsuccessfulTouches' : "UnTch",
            'Dispossessed' : 'Dispo'
        }, inplace=True)
    if 'Unnamed: 0' in Possession_Loss.columns:
        Possession_Loss.drop(columns=['Unnamed: 0'], inplace=True)
    
    # Read *data_details_Aerial_EPL.csv
    Aerial = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Aerial_{league}.csv') 
    Aerial.rename (columns={
        'Total' : 'TotAD',
        'Won' : 'WonAD',
        'Lost' : 'LostAD'
    }, inplace=True)
    if 'Unnamed: 0' in Aerial.columns:
        Aerial.drop(columns=['Unnamed: 0'], inplace=True)
    
    Offensive_Profile = pd.concat ([Goals, Shots, Dribbles, Possession_Loss, Aerial], axis=1)
    print (f"{league} Offensive Profile Shape: {Offensive_Profile.shape}")
    return Offensive_Profile
    
EPL_Offensive_Profile = Read_Data_Details_Offensive ('EPL', EPL_SpG)
Laliga_Offensive_Profile = Read_Data_Details_Offensive ('Laliga', Laliga_SpG)
SerieA_Offensive_Profile = Read_Data_Details_Offensive ('SerieA', SerieA_SpG)
Bundesliga_Offensive_Profile = Read_Data_Details_Offensive ('Bundesliga', Bundesliga_SpG)
Ligue1_Offensive_Profile = Read_Data_Details_Offensive ('Ligue1', Ligue1_SpG)

# Tạo file ..._Offensive_Profile
# EPL_Offensive_Profile.to_csv ('EPL_Offensive_Profile.csv')
# Laliga_Offensive_Profile.to_csv ('Laliga_Offensive_Profile.csv')
# Bundesliga_Offensive_Profile.to_csv ('Bundesliga_Offensive_Profile.csv')
# SerieA_Offensive_Profile.to_csv ('SerieA_Offensive_Profile.csv')
# Ligue1_Offensive_Profile.to_csv ('Ligue1_Offensive_Profile.csv')

'''Chỉ sử dụng nếu cần merge'''
Football_Offensive_Profile_Full = pd.concat ([EPL_Offensive_Profile, Laliga_Offensive_Profile,
                                           Bundesliga_Offensive_Profile, SerieA_Offensive_Profile,
                                           Ligue1_Offensive_Profile
                                        ], axis=0, ignore_index=True)
# Football_Offensive_Profile_Full.to_csv ('Football_Offensive_Profile_Full.csv')

EPL Offensive Profile Shape: (564, 17)
Laliga Offensive Profile Shape: (587, 17)
SerieA Offensive Profile Shape: (625, 17)
Bundesliga Offensive Profile Shape: (487, 17)
Ligue1 Offensive Profile Shape: (543, 17)


# 3. Create `Defensive_Profile` Dataset

__Lưu ý:__ Đây là dữ liệu thu thập từ WhoScored

In [37]:
info = []
def Read_Data_Details_Defensive (league):
    tackle = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Tackles_{league}.csv')
    interception = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Interception_{league}.csv')
    foul = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Fouls_{league}.csv')
    offside = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Offsides_{league}.csv')
    clearance = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Clearances_{league}.csv')
    blocked = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Blocks_{league}.csv')
    save = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Saves_{league}.csv')
    save = save.reindex (columns=[
        'Total','SixYardBox',
        'PenaltyArea','OutOfBox'
    ])
    Defensive_Profile = pd.concat ([tackle, interception, foul, offside, clearance,
                           blocked, save],axis=1) 
    Defensive_Profile.columns = [
                            'TotTkl', 'DribPast','AttTkl','Intercpt',
                            'Fouled','Fouls','COF','Clr','BlkSh','BlkCr',
                            'BlkPs','TotSav', 'OOBSav', 'SYBSav','PASav'
                        ]
    print (f"{league} Defensive Profile Shape: {Defensive_Profile.shape}")
    return Defensive_Profile

EPL_Defensive_Profile = Read_Data_Details_Defensive ('EPL')
Laliga_Defensive_Profile = Read_Data_Details_Defensive ('Laliga')
Bundesliga_Defensive_Profile = Read_Data_Details_Defensive ('Bundesliga')
SerieA_Defensive_Profile = Read_Data_Details_Defensive ('SerieA')
Ligue1_Defensive_Profile = Read_Data_Details_Defensive ('Ligue1')
    
# Tạo file ..._Defensive_Profile
# EPL_Defensive_Profile.to_csv ('EPL_Defensive_Profile.csv')
# Laliga_Defensive_Profile.to_csv ('Laliga_Defensive_Profile.csv')
# Bundesliga_Defensive_Profile.to_csv ('Bundesliga_Defensive_Profile.csv')
# SerieA_Defensive_Profile.to_csv ('SerieA_Defensive_Profile.csv')
# Ligue1_Defensive_Profile.to_csv ('Ligue1_Defensive_Profile.csv')

Ligue1_Defensive_Profile

EPL Defensive Profile Shape: (564, 15)
Laliga Defensive Profile Shape: (587, 15)
Bundesliga Defensive Profile Shape: (487, 15)
SerieA Defensive Profile Shape: (625, 15)
Ligue1 Defensive Profile Shape: (543, 15)


,TotTkl,DribPast,AttTkl,Intercpt,Fouled,Fouls,COF,Clr,BlkSh,BlkCr,BlkPs,TotSav,OOBSav,SYBSav,PASav
0,1,2,3,2,-,-,-,1,1,-,1,-,-,-,-
1,2,-,2,2,-,-,-,4,2,-,-,-,-,-,-
2,0.4,0.3,0.7,0.3,0.5,0.3,0.3,-,-,-,0.1,-,-,-,-
3,2.5,1,3.5,1,1.5,0.5,-,3,-,-,-,-,-,-,-
4,0.5,0.1,0.5,-,0.3,0.7,0.4,0.1,-,-,0.5,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,-,-,-,-,-,1,-,-,-,-,-,-,-,-,-
539,0.5,-,0.5,0.5,-,1.5,-,-,-,-,-,-,-,-,-
540,0.7,0.2,0.8,0.5,0.7,1.2,-,2.2,0.5,-,-,-,-,-,-
541,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


# Create `Passing_Profile` Dataset

In [38]:
def Read_Data_Details_Passing (league):
    passes = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Passes_{league}.csv')
    key_pass = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Key passes_{league}.csv')
    assists = pd.read_csv (f'Web_Scraping_Files/{league}/data_details_Assists_{league}.csv')
    assists = assists.reindex (columns=[
        'Total','Cross', 'Corner','Throughball',
        'Freekick','Throwin','Other'
    ])
    Passing_Profile = pd.concat ([passes, key_pass, assists],axis=1) 
    Passing_Profile.columns = [
                    'TotPs', 'AccLB','InAccLB','AccSP',
                    'InAccSP','TotKPs','LKPs','SKPs',
                    'TotAss', 'CrAss','CorAss','ThrbAss',
                    'FreAss','ThrInAss','OthAss'
                ]
    print (f"{league} Passing Profile Shape: {Passing_Profile.shape}")
    return Passing_Profile

EPL_Passing_Profile = Read_Data_Details_Passing ('EPL')
Laliga_Passing_Profile = Read_Data_Details_Passing ('Laliga')
Bundesliga_Passing_Profile = Read_Data_Details_Passing ('Bundesliga')
SerieA_Passing_Profile = Read_Data_Details_Passing ('SerieA')
Ligue1_Passing_Profile = Read_Data_Details_Passing ('Ligue1')

# Tạo file ..._Defensive_Profile
# EPL_Passing_Profile.to_csv ('EPL_Passing_Profile.csv')
# Laliga_Passing_Profile.to_csv ('Laliga_Passing_Profile.csv')
# Bundesliga_Passing_Profile.to_csv ('Bundesliga_Passing_Profile.csv')
# SerieA_Passing_Profile.to_csv ('SerieA_Passing_Profile.csv')
# Ligue1_Passing_Profile.to_csv ('Ligue1_Passing_Profile.csv')

EPL_Passing_Profile

EPL Passing Profile Shape: (564, 15)
Laliga Passing Profile Shape: (587, 15)
Bundesliga Passing Profile Shape: (487, 15)
SerieA Passing Profile Shape: (625, 15)
Ligue1 Passing Profile Shape: (543, 15)


,TotPs,AccLB,InAccLB,AccSP,InAccSP,TotKPs,LKPs,SKPs,TotAss,CrAss,CorAss,ThrbAss,FreAss,ThrInAss,OthAss
0,31,0.7,0.7,22.3,7.3,2.3,0.2,2.2,0.6,0.1,-,-,-,-,0.5
1,24.1,0.3,0.5,20.1,3.3,2.3,0.6,1.7,0.5,0.3,0.2,0.1,-,-,0.2
2,19.2,0.4,0.4,14,4.4,1.3,0.1,1.2,0.2,-,-,-,-,-,0.2
3,28.8,1.4,1.2,21,5.1,1.7,0.2,1.5,0.1,-,-,-,-,-,0.1
4,12,0.1,-,7.9,4,0.9,-,0.9,0.1,-,-,-,-,-,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
560,29.7,2,2.1,22.4,3.1,0.1,-,0.1,-,-,-,-,-,-,-
561,1,-,-,1,-,-,-,-,-,-,-,-,-,-,-
562,24,-,0.5,20,3.5,-,-,-,-,-,-,-,-,-,-


# 4. Create `Football_Player_Full` Dataset

In [39]:
def concat_profiles_by_league(profile, offensive, defensive, passing):
    return pd.concat([
        profile,
        offensive.reindex(profile.index),
        defensive.reindex(profile.index),
        passing.reindex(profile.index)
    ], axis=1)
    
EPL_Football_Player_Full = concat_profiles_by_league(
    EPL_Football_Player_Profile,
    EPL_Offensive_Profile,
    EPL_Defensive_Profile,
    EPL_Passing_Profile
)

Laliga_Football_Player_Full = concat_profiles_by_league(
    Laliga_Football_Player_Profile,
    Laliga_Offensive_Profile,
    Laliga_Defensive_Profile,
    Laliga_Passing_Profile
)

Bundesliga_Football_Player_Full = concat_profiles_by_league(
    Bundesliga_Football_Player_Profile,
    Bundesliga_Offensive_Profile,
    Bundesliga_Defensive_Profile,
    Bundesliga_Passing_Profile
)

SerieA_Football_Player_Full = concat_profiles_by_league(
    SerieA_Football_Player_Profile,
    SerieA_Offensive_Profile,
    SerieA_Defensive_Profile,
    SerieA_Passing_Profile
)

Ligue1_Football_Player_Full = concat_profiles_by_league(
    Ligue1_Football_Player_Profile,
    Ligue1_Offensive_Profile,
    Ligue1_Defensive_Profile,
    Ligue1_Passing_Profile
)

EPL_Football_Player_Full.to_csv ('EPL_Football_Player_Full.csv')
Laliga_Football_Player_Full.to_csv ('Laliga_Football_Player_Full.csv')
Bundesliga_Football_Player_Full.to_csv ('Bundesliga_Football_Player_Full.csv')
SerieA_Football_Player_Full.to_csv ('SerieA_Football_Player_Full.csv')
Ligue1_Football_Player_Full.to_csv ('Ligue1_Football_Player_Full.csv')
Football_Player_Full_Dataset = pd.concat ([EPL_Football_Player_Full, 
                                           Laliga_Football_Player_Full,
                                           Bundesliga_Football_Player_Full,
                                           SerieA_Football_Player_Full,
                                           Ligue1_Football_Player_Full], axis=0, ignore_index=True)
Football_Player_Full_Dataset.to_csv ('Football_Player_Full_Dataset.csv')

In [20]:
Football_Player_Full_Dataset.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 68 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Player         3044 non-null   object 
 1   Club           3044 non-null   object 
 2   League         2806 non-null   object 
 3   Age            2805 non-null   object 
 4   Height_all     3036 non-null   object 
 5   National       2595 non-null   object 
 6   Positions      2806 non-null   object 
 7   Main_position  2648 non-null   object 
 8   Foot           2625 non-null   object 
 9   Apps           2806 non-null   object 
 10  Mins           2806 non-null   float64
 11  Goals          2806 non-null   object 
 12  Assists        2806 non-null   object 
 13  Yel            2806 non-null   object 
 14  Red            2806 non-null   object 
 15  PS%            2806 non-null   object 
 16  AerialsWon     2806 non-null   object 
 17  MotM           2806 non-null   object 
 18  Rating  

**Lưu ý:** Đặt file Data_Merging.ipynb folder ngay bên ngoài folder *Web_Scraping_Files* thì chạy được